# Wing Design — Electric V-BAT-Like Tail-Sitter (EDF)

This notebook covers airfoil and wing geometry sizing.
It picks up directly from the converged mass iteration loop
and the constraint diagram outputs.

All numerical values are computed in the code cells below.
Do not hardcode mass, weight, or wing loading here.

Notebook flow:
  1. Imports and design point inputs
  2. Wing area and stall speed
  3. Airfoil selection rationale
  4. NACA 4-digit geometry
  5. Lift curve slope
  6. CL-CD polar
  7. Span, chord, AR, taper
  8. Reynolds number check
  9. Cruise validation
 10. AR trade study
 11. Summary

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

import math
import numpy as np
import yaml

from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import solve_design_point
from conceptual_design import reports
from conceptual_design.plots import (
    plot_airfoil_profile, plot_ar_trade, plot_drag_polar, plot_lift_curve,
)

nb = nb_setup()
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

## 1. Design Point Inputs

Mass and weight come directly from the converged sizing loop.
Wing loading and cruise T/W come from the constraint diagram notebook.
Do not hardcode these values here.

In [ ]:
# -- Re-run the converged design point from config/ (same as every notebook) ---
inp, result = solve_design_point(CONFIG_PATH)
env, mission, aero, ws = inp.env, inp.mission, inp.aero, inp.ws

m_total  = result.m_total_kg
W        = m_total * env.g
V_cruise = mission.V_cruise

print(f"Converged mass  m_total = {m_total:.3f} kg")
print(f"Vehicle weight  W       = {W:.2f} N")

## 2. Wing Area and Stall Speed

The design wing loading comes from the Size Matching Diagram in the
conceptual-design notebook.  Once MTOW and W/S are known, wing area
and span follow directly.

Stall speed check:
$$V_{stall} = \sqrt{\frac{2 \, W/S}{\rho \, C_{L,max}}}$$

In [ ]:
from conceptual_design.forward_flight_power import compute_size_matching_diagram
from conceptual_design.wing_sizing import size_wing

# -- Re-run the size matching diagram to get the design point ----------
smd = compute_size_matching_diagram(
    aero           = aero,
    mission        = mission,
    ff             = inp.ff,
    env            = env,
    eta_propulsive = inp.prop.eta_total,
)

WS_design = smd.WS_design    # [N/m^2]  design wing loading
TW_design = smd.TW_design    # [-]      design thrust-to-weight

# -- Wing geometry ------------------------------------------------------
wing = size_wing(
    MTOW_kg   = m_total,
    WS_design = WS_design,
    TW_design = TW_design,
    aero      = aero,
    mission_V = V_cruise,
    ws        = ws,
    env       = env,
)

# -- Stall speed --------------------------------------------------------
V_stall = math.sqrt(2.0 * WS_design / (env.rho * aero.CL_max))

print(f"Design wing loading : {WS_design:.1f} N/m^2")
print(f"Wing area           : {wing.S_wing:.4f} m^2")
print(f"Wing span           : {wing.b_wing:.4f} m")
print(f"Mean chord (MAC)    : {wing.chord_mean:.4f} m")
print(f"Wing mass (Raymer)  : {wing.mass_wing_kg:.4f} kg")
print(f"Stall speed         : {V_stall:.2f} m/s  (require <= {aero.V_stall:.1f} m/s)")
stall_ok = V_stall <= aero.V_stall * 1.02   # 2 % numerical margin
print(f"Stall constraint    : {'OK' if stall_ok else 'VIOLATED'}")

## 3. Airfoil Selection Rationale

For a tail-sitter EDF operating at Re ~ 3×10⁵ – 8×10⁵ the wing must
satisfy three competing requirements:

| Requirement | Driver |
|---|---|
| Low profile drag | cruise endurance |
| Adequate Cl_max | stall speed at design W/S |
| t/c ≥ 0.09 | spar depth for structural stiffness |

Candidate sections are defined in `config/airfoil_selection.yaml`.
The analysis below compares them and selects the best-fit airfoil.

In [ ]:
from conceptual_design.airfoil_selection import compare_airfoils

# -- Airfoil selection parameters (config/airfoil_selection.yaml) -------
with open(CONFIG_PATH / "airfoil_selection.yaml", "r", encoding="utf-8") as _f:
    _af_cfg = yaml.safe_load(_f)

DESIGNATION  = _af_cfg["designation"]     # e.g. "NACA 2412"
CD0_FUSELAGE = _af_cfg["CD0_fuselage"]    # fuselage drag contribution

# -- Side-by-side comparison of candidate airfoils ----------------------
candidates = ["NACA 0012", "NACA 2412", "NACA 4412", "NACA 2415"]
print("Airfoil comparison  (AR={:.1f}, W/S={:.0f} N/m^2, V={:.0f} m/s)\n".format(
    aero.AR, WS_design, V_cruise))
compare_airfoils(
    candidates   = candidates,
    AR           = aero.AR,
    WS_N_m2      = WS_design,
    V_cruise     = V_cruise,
    V_stall      = aero.V_stall,
    rho          = env.rho,
)
print(f"\nSelected : {DESIGNATION}  (from config/airfoil_selection.yaml)")

## 4. NACA 4-Digit Geometry

The selected airfoil is analysed in full and its upper/lower surface
coordinates are plotted at unit chord.

Thickness distribution (NACA standard):
$$y_t = 5t\bigl(0.2969\sqrt{x} - 0.1260x - 0.3516x^2 + 0.2843x^3 - 0.1015x^4\bigr)$$

In [ ]:
from conceptual_design.airfoil_selection import parse_naca4

M, P, t = parse_naca4(DESIGNATION)
print(f"{DESIGNATION}")
print(f"  Max camber       M = {M:.4f}  ({M*100:.1f}% chord)")
print(f"  Camber location  P = {P:.2f}   ({P*100:.0f}% chord)")
print(f"  Thickness        t = {t:.4f}  ({t*100:.1f}% chord)")

plot_airfoil_profile(M, P, t, DESIGNATION, FIG_DIR / 'airfoil_profile.png')

## 5. Lift Curve Slope

**2D section** (thin airfoil + thickness correction, Abbott & von Doenhoff):
$$C_{l,\alpha} = 2\pi\,(1 + 0.77\,t)  \quad [\text{per rad}]$$

**3D finite wing** (Prandtl lifting-line):
$$C_{L,\alpha} = \frac{C_{l,\alpha}}{1 + C_{l,\alpha}/(\pi\,AR\,e)}$$

In [ ]:
from conceptual_design.airfoil_selection import (
    section_Cl_alpha, wing_CL_alpha, oswald_efficiency, section_alpha_L0,
)

Cl_a_2D  = section_Cl_alpha(t)
e_oswald = oswald_efficiency(aero.AR)
CL_a_3D  = wing_CL_alpha(Cl_a_2D, aero.AR, e_oswald)
aL0_deg  = math.degrees(section_alpha_L0(M, P))

plot_lift_curve(Cl_a_2D, CL_a_3D, aL0_deg, aero.AR, DESIGNATION,
                FIG_DIR / 'lift_curve.png')

print(f"2D  Cl_alpha = {Cl_a_2D:.4f} /rad  ({math.degrees(Cl_a_2D):.4f} /deg)")
print(f"3D  CL_alpha = {CL_a_3D:.4f} /rad")
print(f"    alpha_L0 = {aL0_deg:.3f} deg")
print(f"    Oswald e = {e_oswald:.4f}")

## 6. CL–CD Polar

Parabolic drag polar for the 3D wing:
$$C_D = C_{D0} + k\,C_L^2, \qquad k = \frac{1}{\pi\,AR\,e}$$

$C_{D0}$ includes both the wing profile drag and the fuselage/misc
contribution loaded from `config/airfoil_selection.yaml`.

In [ ]:
from conceptual_design.airfoil_selection import section_Cd0

Cd0_section = section_Cd0(t)
CD0_total   = Cd0_section + CD0_FUSELAGE
k_induced   = 1.0 / (math.pi * aero.AR * e_oswald)

# Cruise operating point on the polar
q_cruise  = 0.5 * env.rho * V_cruise**2
CL_cruise = WS_design / q_cruise
CD_cruise = CD0_total + k_induced * CL_cruise**2
LD_cruise = CL_cruise / CD_cruise

plot_drag_polar(CD0_total, k_induced, aero.CL_max, CL_cruise, CD_cruise,
                LD_cruise, DESIGNATION, FIG_DIR / 'cl_cd_polar.png')

print(f"Cd0 section  : {Cd0_section:.5f}")
print(f"CD0 total    : {CD0_total:.5f}  (incl. {CD0_FUSELAGE:.3f} fuselage)")
print(f"k (induced)  : {k_induced:.5f}")
print(f"CL cruise    : {CL_cruise:.4f}")
print(f"L/D cruise   : {LD_cruise:.2f}")

## 7. Span, Chord, AR, Taper

Geometry relationships for a simple rectangular planform
(taper ratio λ = 1, no sweep):

$$b = \sqrt{AR \cdot S}, \qquad c_{MAC} = \frac{S}{b} = \sqrt{\frac{S}{AR}}$$

These are the same values computed by `size_wing()`; this section
makes the geometry parameters explicit and checks the 3 m size constraint.

In [ ]:
reports.print_planform_summary(wing, aero, ws)

## 8. Reynolds Number Check

$$Re = \frac{\rho \, V \, c}{\mu}$$

At ISA sea level, dynamic viscosity μ = 1.789×10⁻⁵ Pa·s.
The airfoil empirical data used above is calibrated for
Re ~ 3×10⁵ – 1×10⁶.  Values below 2×10⁵ indicate laminar-separation
risk and the Cl_max estimate becomes non-conservative.

In [ ]:
Re_cruise, Re_stall = reports.print_reynolds_check(
    env.rho, V_cruise, V_stall, wing.chord_mean)

## 9. Cruise Validation

Run the full airfoil analysis for the selected section and confirm
that all three design constraints are satisfied:

1. Wing loading ≤ stall limit
2. t/c ≥ 0.09 (spar depth)
3. 2D Cl_max has ≥ 5% headroom above the 3D CL_max requirement

In [ ]:
from conceptual_design.airfoil_selection import analyse_airfoil, write_outputs

af = analyse_airfoil(
    designation  = DESIGNATION,
    AR           = aero.AR,
    WS_N_m2      = WS_design,
    V_cruise     = V_cruise,
    V_stall      = aero.V_stall,
    rho          = env.rho,
    CD0_fuselage = CD0_FUSELAGE,
)
af.print_summary()

# -- Write generated outputs (out/airfoil.yaml and out/airfoils/*.dat) --
write_outputs(af, out_dir=OUT_PATH, n_coords=int(_af_cfg["n_coords"]),
              chord=wing.chord_mean)

## 10. Aspect Ratio Trade Study

Aspect ratio drives the induced-drag factor k and therefore cruise L/D.
Higher AR improves L/D (endurance) but increases wing mass and span.
This sweep shows the trade-off at fixed wing loading and MTOW.

In [ ]:
AR_values = np.linspace(4.0, 10.0, 60)
plot_ar_trade(aero.AR, AR_values, CD0_total, CL_cruise, W, WS_design,
              m_total, V_cruise, env.rho, ws, FIG_DIR / 'ar_trade.png')

## 11. Summary

Final wing and airfoil design card for the electric V-BAT tail-sitter.

In [ ]:
reports.print_wing_card(af, wing, aero, ws, M, P, WS_design, V_stall,
                        Re_cruise, Re_stall)